## Archive photo attachments

This notebook demonstrates how to archive photo attachments with a watermark of latitude and longitude from Exif GPS info using the Pillow Library.  The original attachments are replaced with a generated thumbnail.

To use this notebook, you need to update variables at the top of the script, FEATURE_LAYER_ITEM_ID to the item id of the feature layer to archive attachments for and FEATURE_LAYER_ID to specify the specific layer

This notebook assumes that:

- Change tracking is enabled for the feature layer used.  Ensure "Keep track of changes to the data" is checked in your ArcGIS Online/Enterprise feature layer item settings.
- You are using Enterprise 10.8.1+ or ArcGIS Online

For more information see:

- ArcGIS Python API Layer Attachments - https://developers.arcgis.com/python/guide/using-attachments-with-feature-layers/
- Pillow Library - https://pillow.readthedocs.io/en/stable/index.html

In [ ]:
import os
import shutil
import tempfile
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
from datetime import datetime as dt
from PIL import ExifTags
from PIL import Image
from PIL import ImageDraw
from PIL import ImageOps


# Using built-in user
# For information on different authentication schemes see
# https://developers.arcgis.com/python/guide/working-with-different-authentication-schemes/
# and for protecting your credentials see
# https://developers.arcgis.com/python/guide/working-with-different-authentication-schemes/#protecting-your-credentials
gis = GIS("home", verify_cert=True)

# The item id of the feature layer to process attachments for
FEATURE_LAYER_ITEM_ID = "7a28f3f872bd4b55bfd7965bf73a46b6"
# The id of the layer to use
FEATURE_LAYER_ID = 0
# Local file used to store the previous layer server generation
SERVER_GENERATION_FILE = "archive_photo_attachments_servergen.txt"


# Converts latitude and longitude from Exif GPS info in degrees, minutes and seconds to decimal degrees
def convert_dms_to_dd(gps_info) -> tuple[float, float]:

    if gps_info is None:
        return (0, 0)

    lat_degrees = gps_info[ExifTags.GPS.GPSLatitude][0]
    lat_minutes = gps_info[ExifTags.GPS.GPSLatitude][1]
    lat_seconds = gps_info[ExifTags.GPS.GPSLatitude][2]
    lon_degrees = gps_info[ExifTags.GPS.GPSLongitude][0]
    lon_minutes = gps_info[ExifTags.GPS.GPSLongitude][1]
    lon_seconds = gps_info[ExifTags.GPS.GPSLongitude][2]

    lat = float(lat_degrees + lat_minutes / 60 + lat_seconds / 3600)
    lon = float(lon_degrees + lon_minutes / 60 + lon_seconds / 3600)

    if gps_info[ExifTags.GPS.GPSLatitudeRef] == "S":
        lat = -lat
    if gps_info[ExifTags.GPS.GPSLongitudeRef] == "W":
        lon = -lon

    return (lat, lon)


# Watermarks an image with decimal degree latitude and longitude from Exif data
def add_dd_watermark(image_path):

    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image)

    # Get the image's Exif data, convert the lat and long to decimal degree and format the string for the watermark
    exif = image.getexif()
    gps_ifd = exif.get_ifd(ExifTags.IFD.GPSInfo)
    (lat, lon) = convert_dms_to_dd(gps_ifd)
    watermark_text = f"Latitude: {lat:.4f} Longitude: {lon:.4f}"

    draw = ImageDraw.Draw(image)

    # Draw the watermark in the top left corner of the image in white
    # For more drawing options see: https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html#PIL.ImageDraw.ImageDraw.text
    # See https://pillow.readthedocs.io/en/stable/reference/ImageFont.html for information on how to change the font, including font size
    text_xy_position = (10, 10)
    rgb_fill_color = (255, 255, 255)
    draw.text(text_xy_position, watermark_text, fill=rgb_fill_color)
    image.save(image_path)
    # Optional call to show the photo with the watermark
    image.show()


# Creates a thumbnail image in the same directory using the original name with '_thumbnail' suffix
def create_attachment_thumbnail(image_path, max_size=(256, 256)):

    thumbnail_path = os.path.splitext(image_path)[0] + "_thumbnail" + os.path.splitext(image_path)[1]
    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image)

    thumbnail_image = image.copy()
    thumbnail_image.thumbnail(max_size)
    thumbnail_image.save(thumbnail_path)

    return thumbnail_path


# Read previous server generation from file (if available)
previous_server_generation = None
if os.path.exists(SERVER_GENERATION_FILE):
    with open(SERVER_GENERATION_FILE, "r", encoding="utf-8") as generation_file:
        generation_text = generation_file.read().strip()
        if generation_text:
            previous_server_generation = int(generation_text)

layer_item = gis.content.get(FEATURE_LAYER_ITEM_ID)
layer = layer_item.layers[FEATURE_LAYER_ID]
flc = FeatureLayerCollection.fromitem(layer_item)

# Extract only inserted features for this one layer since the previous generation
extract_kwargs = {
    "layers": [FEATURE_LAYER_ID],
    "return_inserts": True,
    "return_updates": False,
    "return_deletes": False,
    "return_ids_only": True,
}
if previous_server_generation is not None:
    extract_kwargs["layer_servergen"] = [
        {"id": FEATURE_LAYER_ID, "serverGen": previous_server_generation}
    ]
deltas = flc.extract_changes(**extract_kwargs)

# Persist the latest generation for the next run
layer_server_gens = deltas.get("layerServerGens", [])
if layer_server_gens:
    current_server_generation = layer_server_gens[0].get("serverGen")
    if current_server_generation is not None:
        with open(SERVER_GENERATION_FILE, "w", encoding="utf-8") as generation_file:
            generation_file.write(str(current_server_generation))

# Collect object IDs for added features only from the first (and only) layer edit
edits = deltas.get("edits", [])
first_edit = edits[0] if edits else {}
object_ids = first_edit.get("objectIds", {})
added_oids = object_ids.get("adds", [])

if not added_oids:
    print("No added features found since the previous server generation. Stopping processing.")
else:
    # Create temporary output directories
    temporary_attachment_output = tempfile.TemporaryDirectory()
    temporary_zip_output = tempfile.TemporaryDirectory()
    temporary_zip_output_name = os.path.join(
        temporary_zip_output.name,
        "zipped_photos_{}".format(dt.now().strftime("%Y-%m-%d %H-%M-%S")),
    )

    # Find all attachments tied to the newly added features
    attachment_infos = [
        attachment
        for attachment in layer.attachments.search(where="1=1")
        if (attachment.get("PARENTOBJECTID") or attachment.get("parentObjectId")) in added_oids
    ]

    # For each attachment: watermark, create thumbnail, replace original on the feature
    for attachment_info in attachment_infos:
        parent_oid = attachment_info.get("PARENTOBJECTID") or attachment_info.get("parentObjectId")
        attachment_id = attachment_info.get("ID") or attachment_info.get("id")

        attachment_dir = os.path.join(
            temporary_attachment_output.name, str(parent_oid), str(attachment_id)
        )
        os.makedirs(attachment_dir, exist_ok=True)

        downloaded_paths = layer.attachments.download(
            oid=parent_oid, attachment_id=attachment_id, save_path=attachment_dir
        )
        downloaded_path = downloaded_paths[0] if isinstance(downloaded_paths, list) else downloaded_paths

        add_dd_watermark(downloaded_path)
        thumbnail_path = create_attachment_thumbnail(downloaded_path)

        layer.attachments.delete(oid=parent_oid, attachment_id=attachment_id)
        layer.attachments.add(oid=parent_oid, file_path=thumbnail_path)

    # Create a zip archive with the watermarked images and thumbnails
    output_zip = shutil.make_archive(
        temporary_zip_output_name, "zip", temporary_attachment_output.name
    )

    # Create a new Image Collection item with the archived watermarked images
    item_props = {
        "type": "Image Collection",
        "typeKeywords": ["Image", "Image Collection", "Photo locations"],
        "description": "Images with watermarks",
        "title": "Images with watermarks",
        "tags": [],
        "snippet": "Images with watermarks",
    }
    root_folder = gis.content.folders.get()
    add_job = root_folder.add(item_properties=item_props, file=output_zip)
    new_item = add_job.result() if hasattr(add_job, "result") else add_job

    # Clean up temporary files
    os.remove(output_zip)
    temporary_attachment_output.cleanup()
    temporary_zip_output.cleanup()

    display(new_item)
